# 08 — DREES Panorama des retraites 2025

Analyse multi-régimes : effectifs, âge de départ, masses financières, pensions mensuelles.

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sqlalchemy as sa
from src.db import engine

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 10
conn = engine().connect()

## 1. Effectifs tous régimes H/F/T (2004–2023)

In [ ]:
eff = pd.read_sql("""
    SELECT annee, genre_code, valeur
    FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur = 'NB_RETRAITES_TOUS_REGIMES'
    ORDER BY annee, genre_code
""", conn)

eff_p = eff.pivot(index='annee', columns='genre_code', values='valeur')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for gc, color in [('F', 'salmon'), ('H', 'steelblue'), ('T', 'gray')]:
    if gc in eff_p.columns:
        ax1.plot(eff_p.index, eff_p[gc] / 1e3, label={'F': 'Femmes', 'H': 'Hommes', 'T': 'Ensemble'}[gc], color=color)

ax1.set_title('Effectifs tous régimes (milliers)', fontweight='bold')
ax1.set_ylabel('Milliers'); ax1.legend(); ax1.grid(alpha=0.3)

# Flux (nouveaux retraités et variation)
flux = pd.read_sql("""
    SELECT annee, indicateur, valeur
    FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur IN ('NOUVEAUX_RETRAITES','EVOL_NB_RETRAITES_PCT')
    ORDER BY annee, indicateur
""", conn)

nv = flux[flux['indicateur'] == 'NOUVEAUX_RETRAITES']
ev = flux[flux['indicateur'] == 'EVOL_NB_RETRAITES_PCT']

ax2b = ax2.twinx()
ax2.bar(nv['annee'], pd.to_numeric(nv['valeur'], errors='coerce'), alpha=0.6, color='steelblue', label='Nouveaux (k)')
ax2b.plot(ev['annee'], pd.to_numeric(ev['valeur'], errors='coerce'), color='red', lw=1.5, label='Évol. %')
ax2.set_title('Flux de nouveaux retraités et évolution (%)', fontweight='bold')
ax2.set_ylabel('Milliers'); ax2b.set_ylabel('Évol. %')
ax2.legend(loc='upper left'); ax2b.legend(loc='upper right'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/08_effectifs_drees.png', bbox_inches='tight')
plt.show()

## 2. Âge conjoncturel de départ et réformes

In [ ]:
age_dep = pd.read_sql("""
    SELECT annee, genre_code, valeur
    FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur = 'AGE_CONJONCTUREL_DEPART'
    ORDER BY annee, genre_code
""", conn)

reformes = {1993: 'Balladur', 2003: 'Fillon', 2010: 'Woerth', 2014: 'Touraine', 2023: 'Borne'}

fig, ax = plt.subplots(figsize=(13, 5))
for gc, color, label in [('F', 'salmon', 'Femmes'), ('H', 'steelblue', 'Hommes'), ('T', 'dimgray', 'Ensemble')]:
    sub = age_dep[age_dep['genre_code'] == gc]
    ax.plot(sub['annee'], pd.to_numeric(sub['valeur'], errors='coerce'), color=color, label=label, lw=2)

for yr, nom in reformes.items():
    if 2004 <= yr <= 2023:
        ax.axvline(yr, color='red', ls='--', lw=0.8, alpha=0.7)
        ax.text(yr + 0.1, ax.get_ylim()[0] + 0.05, nom, rotation=90, fontsize=7, color='red', va='bottom')

ax.set_title('Âge conjoncturel de départ à la retraite — tous régimes', fontweight='bold')
ax.set_ylabel('Âge (ans)'); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(2003, 2024); ax.set_ylim(60, 64.5)
plt.tight_layout()
plt.savefig('reports/08_age_depart.png', bbox_inches='tight')
plt.show()

## 3. Masses financières et part du PIB (1990–2023)

In [ ]:
masses = pd.read_sql("""
    SELECT annee, indicateur, valeur
    FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur IN ('PRESTATIONS_TOTAL_MDS_EUR','PART_TOTAL_PIB_PCT',
                         'PRESTATIONS_DD_MDS_EUR','PRESTATIONS_DDE_MDS_EUR')
    ORDER BY annee, indicateur
""", conn)

masses_p = masses.pivot(index='annee', columns='indicateur', values='valeur').apply(pd.to_numeric, errors='coerce')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if 'PRESTATIONS_TOTAL_MDS_EUR' in masses_p:
    ax1.fill_between(masses_p.index, masses_p['PRESTATIONS_TOTAL_MDS_EUR'], alpha=0.3, color='steelblue')
    ax1.plot(masses_p.index, masses_p['PRESTATIONS_TOTAL_MDS_EUR'], color='steelblue', label='Total')
if 'PRESTATIONS_DD_MDS_EUR' in masses_p:
    ax1.plot(masses_p.index, masses_p['PRESTATIONS_DD_MDS_EUR'], color='navy', ls='--', label='Droits directs')
ax1.set_title('Prestations de retraite (milliards €)', fontweight='bold')
ax1.set_ylabel('Mds €'); ax1.legend(); ax1.grid(alpha=0.3)

if 'PART_TOTAL_PIB_PCT' in masses_p:
    ax2.plot(masses_p.index, masses_p['PART_TOTAL_PIB_PCT'], color='darkorange', lw=2)
    ax2.fill_between(masses_p.index, masses_p['PART_TOTAL_PIB_PCT'], alpha=0.2, color='darkorange')
ax2.set_title('Part des retraites dans le PIB (%)', fontweight='bold')
ax2.set_ylabel('%'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('reports/08_masses_pib.png', bbox_inches='tight')
plt.show()

## 4. Niveau de pension — CNAV vs tous régimes (DREES Fiche 05)

In [ ]:
comp = pd.read_sql("""
    SELECT annee, pension_moy_cnav, pension_brute_drees_f, pension_brute_drees_h, pension_brute_drees_t
    FROM rpt.v_CompaisonRegimes
    WHERE annee BETWEEN 2004 AND 2023
    ORDER BY annee
""", conn)

fig, ax = plt.subplots(figsize=(13, 5))

for col, color, label in [
    ('pension_moy_cnav',      'navy',       'CNAV (total droits)'),
    ('pension_brute_drees_t', 'steelblue',  'DREES tous régimes (T)'),
    ('pension_brute_drees_f', 'salmon',     'DREES tous régimes (F)'),
    ('pension_brute_drees_h', 'cornflowerblue', 'DREES tous régimes (H)'),
]:
    vals = pd.to_numeric(comp[col], errors='coerce')
    if vals.notna().any():
        ax.plot(comp['annee'], vals, color=color, label=label, lw=2)

ax.set_title('Pension mensuelle brute : CNAV vs tous régimes (€ courants)', fontweight='bold')
ax.set_ylabel('€/mois'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/08_pension_cnav_vs_drees.png', bbox_inches='tight')
plt.show()

print("\nDernières valeurs (2023) :")
print(comp[comp['annee'] == 2023].to_string(index=False))

## 5. Pension des nouveaux retraités vs ensemble (euros constants 2023)

In [ ]:
nv_ret = pd.read_sql("""
    SELECT annee, indicateur, valeur
    FROM ext.DREES_PensionsMultiRegimes
    WHERE indicateur IN ('PENSION_BRUTE_NOUVEAUX_EUR2023','PENSION_BRUTE_ENSEMBLE_EUR2023')
    ORDER BY annee, indicateur
""", conn)

nv_p = nv_ret.pivot(index='annee', columns='indicateur', values='valeur').apply(pd.to_numeric, errors='coerce')

fig, ax = plt.subplots(figsize=(13, 5))
if 'PENSION_BRUTE_NOUVEAUX_EUR2023' in nv_p:
    ax.plot(nv_p.index, nv_p['PENSION_BRUTE_NOUVEAUX_EUR2023'], color='tomato', lw=2, label='Primo-liquidants')
if 'PENSION_BRUTE_ENSEMBLE_EUR2023' in nv_p:
    ax.plot(nv_p.index, nv_p['PENSION_BRUTE_ENSEMBLE_EUR2023'], color='steelblue', lw=2, label='Ensemble retraités')

ax.axhline(ax.get_ylim()[0] if len(ax.lines) else 1500, color='gray', lw=0.5)
ax.set_title('Pension brute mensuelle moyenne (€ constants 2023)\nNouveaux retraités vs ensemble', fontweight='bold')
ax.set_ylabel('€2023/mois'); ax.legend(); ax.grid(alpha=0.3)
ax.text(2018, nv_p.iloc[0, 0] if len(nv_p) else 1600, 'nd = rupture\nLURA 2017-2019', ha='center', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig('reports/08_nouveaux_retraites.png', bbox_inches='tight')
plt.show()

In [ ]:
conn.close()